# DMOSpeech 2: Live 1,000-Utterance GPU Re-test Benchmark
## Full Re-synthesis & Evaluation on NVIDIA CUDA GPU (Tesla T4 / A100)

**Author:** Rhenel Jhon Sajol (Master's Thesis 2026)  
**Scope:** All 500 Speakers across Dev (100), Calib (200), and Test (200) splits  
**Total Evaluated:** 1,000 Condition Pairs (500 Clean + 500 Noisy15) &bull; 4,000 Candidate Evaluations  
**Estimated Runtime:** ~12–15 minutes on Tesla T4 GPU (or ~5–7 minutes on A100 GPU)  

---
### Instructions:
1. Set runtime to **GPU** (`Runtime > Change runtime type > T4 GPU or A100 GPU`).
2. Click **Runtime > Run all** (or press `Ctrl + F9`).

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

### Step 2: Clone Thesis Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists('Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git
%cd /content/Shonget-DMOSpeech2
!git pull origin main

# Install complete package dependencies
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pandas pyarrow scipy jieba accelerate huggingface_hub


### Step 3: Fast Download Checkpoints from HuggingFace
Downloads `model_85000.pt` (4.0 GB) and `model_1500.pt` (270 MB) directly into `ckpts/`.

In [ ]:
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

### Step 4: Download Seed-TTS Evaluation Dataset (Audio Prompts)
Downloads the reference audio prompt wav files from HuggingFace into `data/seed-tts-eval/`.

In [ ]:
import os
from huggingface_hub import snapshot_download

# Check if real audio binary files exist (Git LFS pointer files are only ~130 bytes)
test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'
needs_download = not os.path.exists(test_file) or os.path.getsize(test_file) < 1000

if needs_download:
    print("Downloading Seed-TTS dataset audio files via huggingface_hub (resolving Git LFS binaries)...")
    snapshot_download(
        repo_id="zhaochenyang20/seed-tts-eval",
        repo_type="dataset",
        local_dir="data/seed-tts-eval"
    )
    print("Download complete!")
else:
    print("Seed-TTS audio binaries verified!")

if os.path.exists(test_file):
    sz = os.path.getsize(test_file)
    print(f"Sample file size: {sz / 1024:.1f} KB (Valid audio binary ready)")
!ls -lh data/seed-tts-eval/en/prompt-wavs | head -n 5

### Step 5: Launch Live 1,000-Utterance GPU Benchmark
Runs the full re-test across all 500 speakers (1,000 clean & noisy pairs / 4,000 candidate evaluations) on CUDA GPU.

In [ ]:
# Execute high-speed GPU benchmark for all 1,000 pairs across all 500 speakers
!python "Handover Scheduling/colab/run_gpu_expanded_benchmark.py" --device cuda --scope all1000

### Step 6: View Generated Results Table
Loads and displays the re-tested summary metrics.

In [ ]:
import pandas as pd
import numpy as np

csv_path = "Handover Scheduling/colab/results/gpu_grid_all1000.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"SUCCESS: Re-tested {len(df)} evaluations across {df['speaker_id'].nunique()} speakers!")
    summary = df.groupby(["teacher_step", "condition"])[["nfe", "wer", "similarity", "utmos", "is_failure", "synthesis_time", "rtf"]].mean()
    display(summary)
else:
    print(f"File {csv_path} not found. Run Step 5 first.")